In [13]:
reactions = [
    "CC(=O)O.CCO>[H+].[Cl-]>CC(=O)OCC.O",
    "C=C.[H][H]>[Pd]>CC",
    "c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O"
]

def parse_reaction(rxn):
    a = rxn.split(">")
    if len(a) != 3:
        raise ValueError("reaction must contain exactly two > separators")
    d = {}
    for k, s in zip(["reactants", "reagents", "products"], a):
        d[k] = [] if s == "" else s.split(".")
    return d

for r in reactions:
    d = parse_reaction(r)
    for k in d:
        print(k, len(d[k]), d[k])
    print()

reactants 2 ['CC(=O)O', 'CCO']
reagents 2 ['[H+]', '[Cl-]']
products 2 ['CC(=O)OCC', 'O']

reactants 2 ['C=C', '[H][H]']
reagents 1 ['[Pd]']
products 1 ['CC']

reactants 2 ['c1ccccc1', 'O=[N+]([O-])O']
reagents 0 []
products 2 ['c1ccccc1[N+](=O)[O-]', 'O']



A.1:Reaction 1 (acetic acid + ethanol -> ethyl acetate + water, catalysed by [H+]) is the esterification. Reaction 3 has an empty reagent field (>>), meaning no catalyst or reagent is specified for that transformation (the benzene nitration is written with reactants going directly to products).

In [14]:
import numpy as np

E = np.array([[2, 0, -1, 0],
              [6, 0,  0, -2],
              [0, 2, -2, -1]], float)

u, s, vt = np.linalg.svd(E)
x = vt[-1]
x = x / x[0]
m = 2
c = np.rint(x * m).astype(int)

print("singular values:", s)
print("rank:", np.linalg.matrix_rank(E), "nullity:", E.shape[1] - np.linalg.matrix_rank(E))
print("x:", x)
print("coefficients:", c)
print("E @ x:", E @ x)

singular values: [6.62561256 3.00720721 1.02857332]
rank: 3 nullity: 1
x: [1.  3.5 2.  3. ]
coefficients: [2 7 4 6]
E @ x: [-4.44089210e-16 -1.77635684e-15 -8.88178420e-16]


A.2:A one-dimensional null space means there is one independent stoichiometric balancing relationship, giving 2 C₂H₆ + 7 O₂ → 4 CO₂ + 6 H₂O.




In [15]:
import numpy as np

A = np.zeros((4, 4))
for i in range(3):
    A[i, i + 1] = A[i + 1, i] = 1.0

w = np.linalg.eigvalsh(A)[::-1]
e = 2 * w[:2].sum()

print("eigenvalues:", w)
print("degree:", A.sum(axis=1).astype(int))
print("E_pi = 4 alpha +", e, "beta")
print("delocalisation =", e - 4, "beta")

eigenvalues: [ 1.61803399  0.61803399 -0.61803399 -1.61803399]
degree: [1 2 2 1]
E_pi = 4 alpha + 4.47213595499958 beta
delocalisation = 0.4721359549995796 beta


A.3:Butadiene has a delocalisation stabilization of about \(0.472|\beta|\), smaller than benzene's \(2|\beta|\), showing that benzene has substantially greater π-electron delocalisation.

In [16]:
import numpy as np

A = np.zeros((6, 6))
for i in range(6):
    A[i, (i + 1) % 6] = A[(i + 1) % 6, i] = 1.0

At = A + np.eye(6)
P = At / At.sum(axis=1, keepdims=True)

H = np.random.default_rng(1).normal(size=(6, 3))

v = np.linalg.eigvals(P)
v = np.sort(np.abs(v))[::-1]
mu = v[1]

print("eigenvalue moduli:", v)
print("mu:", mu)

for k in [0, 1, 2, 4, 8, 16]:
    Z = H.copy()
    for _ in range(k):
        Z = P @ Z
    d = np.max(np.linalg.norm(Z - Z.mean(axis=0), axis=1))
    print(k, d)

eigenvalue moduli: [1.00000000e+00 6.66666667e-01 6.66666667e-01 3.33333333e-01
 7.91848035e-17 3.92523115e-17]
mu: 0.6666666666666667
0 1.2413880520727198
1 0.5368253372377662
2 0.3459101833698203
4 0.1548726398897235
8 0.030667573455258054
16 0.0011967999497354295


A.4:
The deviation rapidly approaches zero as layers increase, showing oversmoothing; therefore only a modest number of message-passing layers should be used for molecules of about a dozen heavy atoms.

In [17]:
import numpy as np

X = np.array([[78.1, 2.3],
              [92.1, 1.7],
              [106.2, 2.8],
              [120.2, 0.7],
              [134.2, 3.1],
              [148.2, 2.4]])

def pca(X):
    Xc = X - X.mean(axis=0)
    C = Xc.T @ Xc / len(X)
    w, v = np.linalg.eigh(C)
    j = np.argsort(w)[::-1]
    w, v = w[j], v[:, j]
    for i in range(v.shape[1]):
        q = np.argmax(np.abs(v[:, i]))
        if v[q, i] < 0:
            v[:, i] *= -1
    return C, w, w / w.sum(), v

C, w, f, v = pca(X)
Z = (X - X.mean(axis=0)) / X.std(axis=0)
Cz, wz, fz, vz = pca(Z)

print("centered covariance:\n", C)
print("eigenvalues:", w)
print("fraction:", f)
print("PC1:", v[:, 0])

print("\nstandardized covariance:\n", Cz)
print("eigenvalues:", wz)
print("fraction:", fz)
print("PC1:", vz[:, 0])

centered covariance:
 [[573.53555556   3.03888889]
 [  3.03888889   0.61888889]]
eigenvalues: [573.55167411   0.60277034]
fraction: [0.99895016 0.00104984]
PC1: [0.99998593 0.00530402]

standardized covariance:
 [[1.         0.16129775]
 [0.16129775 1.        ]]
eigenvalues: [1.16129775 0.83870225]
fraction: [0.58064887 0.41935113]
PC1: [0.70710678 0.70710678]


A
NSWER-5 :
The centred PCA is dominated by molar mass because its numerical scale is much larger, whereas standardised PCA gives both descriptors equal scale; therefore the standardised PCA is the more appropriate analysis when the two descriptors are meant to contribute comparably.